# Advanced Data Attributes — Tutorial-Style Problems with Solutions

In this notebook we are going to continue working with **data attributes**.

The focus is still on the distinction between:

- **class attributes**
- **instance attributes**

But this time we will push the ideas further.

Instead of jumping directly into large exercises, we will build each problem in small steps:

1. create a class,
2. inspect its state,
3. create instances,
4. modify one thing,
5. inspect again,
6. explain what Python is doing,
7. solve a more realistic problem.

This is intentionally similar to an exploratory tutorial.

The goal is not just to know *what* happens.

The goal is to become comfortable predicting **where an attribute lives**, **where Python finds it**, and **what will happen after an assignment or deletion**.

We will repeatedly use `__dict__`.

Remember:

- a class has its own namespace,
- an instance can have its own namespace,
- these namespaces are separate,
- attribute lookup can make values *appear* to belong to an instance even when they actually live on the class.

Let's start with a small utility that will make the examples easier to read.

In [1]:
def show_state(label, obj):
    print(f"--- {label} ---")
    print("type:", type(obj).__name__)
    print("__dict__:", getattr(obj, "__dict__", "<no __dict__>"))
    print()

# Problem 1 — A class default that changes after instances already exist

Let's create a class with a class attribute:

In [2]:
class Subscription:
    monthly_price = 10

At this point `monthly_price` belongs to the **class**.

Let's verify that:

In [3]:
Subscription.__dict__["monthly_price"]

10

Now let's create two instances:

In [4]:
basic = Subscription()
premium = Subscription()

We have not assigned any attributes to either instance.

So their instance dictionaries should be empty:

In [5]:
basic.__dict__, premium.__dict__

({}, {})

Yet both instances can access `monthly_price`:

In [6]:
basic.monthly_price, premium.monthly_price

(10, 10)

So the first question is:

> If neither instance contains `monthly_price`, where is Python finding it?

The answer is the class.

Now let's make the problem slightly more interesting.

Suppose the company changes its default monthly price:

In [7]:
Subscription.monthly_price = 12

Before running the next cell, predict the result.

Will existing instances still show `10` because they were created earlier?

Or will they see the new class value?

In [8]:
basic.monthly_price, premium.monthly_price

(12, 12)

### Solution

Both instances see `12`.

The important idea is that the instances did **not copy** the class attribute when they were created.

Their local dictionaries are still empty:

In [9]:
basic.__dict__, premium.__dict__

({}, {})

Python looks for `monthly_price` on the instance.

It does not find it there.

So it continues to the class and finds the current class value.

This means a class attribute can act as a **live shared default**.

# Problem 2 — One object stops following the shared default

Let's continue with the same class.

Right now:

In [10]:
Subscription.monthly_price, basic.monthly_price, premium.monthly_price

(12, 12, 12)

Suppose `premium` has a special negotiated price.

We assign:

In [11]:
premium.monthly_price = 8

That assignment looks simple, but something important just happened.

Let's inspect both instance dictionaries:

In [12]:
basic.__dict__, premium.__dict__

({}, {'monthly_price': 8})

Now `premium` contains its own `monthly_price`.

So let's compare the three values:

In [13]:
Subscription.monthly_price, basic.monthly_price, premium.monthly_price

(12, 12, 8)

### What happened?

`basic` has no local value, so it still reads the class value.

`premium` *does* have a local value, so Python finds that first.

The instance attribute is therefore **shadowing** the class attribute.

Now let's make the class default change again:

In [14]:
Subscription.monthly_price = 15

Predict the result before running:

In [15]:
basic.monthly_price, premium.monthly_price

(15, 8)

### Solution

`basic` sees `15`.

`premium` still sees `8`.

The class changed, but `premium` no longer depends on that class attribute during lookup because its own namespace already contains the same name.

# Problem 3 — Revealing the class attribute again

We currently have an instance override:

In [16]:
premium.__dict__

{'monthly_price': 8}

Suppose the special price expires.

We do **not** want to copy the current class price into the instance.

Instead, we want the instance to go back to following the shared default.

How can we do that?

The key is to remove the **instance attribute**.

Let's do that:

In [17]:
del premium.monthly_price

Now inspect the instance:

In [18]:
premium.__dict__

{}

The local override is gone.

But does `premium.monthly_price` still work?

In [19]:
premium.monthly_price

15

### Solution

Yes.

Deleting the local attribute did not delete the class attribute.

It only removed the first place where Python was finding the name.

Now lookup falls through to the class again.

In [20]:
assert premium.monthly_price == Subscription.monthly_price == 15

# Problem 4 — `getattr` and `setattr` with dynamic field names

So far we have used dotted notation:

In [21]:
basic.monthly_price

15

But sometimes the name of the attribute is stored in a variable.

For example:

In [22]:
field_name = "monthly_price"

In that case we can use `getattr`:

In [23]:
getattr(basic, field_name)

15

Now let's create a new class:

In [24]:
class Server:
    region = "eu-west"
    timeout = 30

And an instance:

In [25]:
server = Server()
server.__dict__

{}

Suppose configuration data arrives as a dictionary:

In [26]:
updates = {
    "timeout": 5,
    "retries": 3,
    "debug": True
}

We do not want to write:

```python
server.timeout = ...
server.retries = ...
server.debug = ...
```

because the keys may only be known at runtime.

So we can use `setattr`:

In [27]:
for name, value in updates.items():
    setattr(server, name, value)

Let's inspect what happened:

In [28]:
server.__dict__

{'timeout': 5, 'retries': 3, 'debug': True}

Notice something subtle.

`timeout` already existed on the class.

But `setattr(server, "timeout", 5)` created a **local instance attribute** with the same name.

So `timeout` is now shadowing the class default.

The other two names did not exist before, so they are simply new instance attributes.

In [29]:
Server.timeout, server.timeout

(30, 5)

We can also safely request an attribute with a fallback:

In [30]:
getattr(server, "hostname", "localhost")

'localhost'

### Solution pattern

Use:

```python
getattr(obj, name)
```

when the attribute name is dynamic.

Use:

```python
getattr(obj, name, default)
```

when a missing attribute is acceptable.

Use:

```python
setattr(obj, name, value)
```

when you need to assign to a dynamically named attribute.

# Problem 5 — The mutable class attribute trap

Now let's look at one of the most important practical problems involving class attributes.

Consider:

In [31]:
class Team:
    members = []

At first glance, this may look like a convenient default.

Let's create two teams:

In [32]:
red = Team()
blue = Team()

Their instance dictionaries are empty:

In [33]:
red.__dict__, blue.__dict__

({}, {})

Now let's add a member to the red team:

In [34]:
red.members.append("Ada")

What do you expect `blue.members` to contain?

In [35]:
red.members, blue.members

(['Ada'], ['Ada'])

Both contain `"Ada"`.

Why?

Let's inspect where `members` actually lives:

In [36]:
"members" in red.__dict__, "members" in blue.__dict__, "members" in Team.__dict__

(False, False, True)

The list lives on the **class**.

Both instances are resolving `members` to the exact same list object.

In [37]:
red.members is blue.members is Team.members

True

This is usually a bug when the list represents per-object state.

So the fix is not to put that mutable object on the class.

Instead, create it for every instance:

In [38]:
class SafeTeam:
    organization = "Open League"

    def __init__(self, name):
        self.name = name
        self.members = []

Now let's test the corrected version:

In [39]:
red = SafeTeam("Red")
blue = SafeTeam("Blue")

red.members.append("Ada")

red.members, blue.members

(['Ada'], [])

And now the two lists are different objects:

In [40]:
red.members is blue.members

False

### Solution principle

A mutable class attribute is appropriate only when you **intentionally want one shared mutable object**.

For per-instance lists, dictionaries, sets, and other mutable containers, initialize them inside `__init__`.

# Problem 6 — A shared counter that accidentally becomes an instance attribute

Let's build a counter.

We want every completed task to increase one number shared by the whole class:

In [41]:
class Task:
    completed = 0

Now imagine we write this method:

In [42]:
class BrokenTask:
    completed = 0

    def mark_done(self):
        self.completed += 1

At first glance that looks reasonable.

Let's create two tasks:

In [43]:
t1 = BrokenTask()
t2 = BrokenTask()

And mark them complete:

In [44]:
t1.mark_done()
t2.mark_done()

Now inspect the class counter:

In [45]:
BrokenTask.completed

0

It is still `0`.

That may be surprising.

Let's inspect the instances:

In [46]:
t1.__dict__, t2.__dict__

({'completed': 1}, {'completed': 1})

Each instance now has its own `completed = 1`.

Why?

The expression:

```python
self.completed += 1
```

behaves conceptually like:

```python
current = self.completed
self.completed = current + 1
```

The read finds the class attribute.

But the assignment writes to the instance.

So the operation accidentally creates a shadow.

To modify shared class state, we should update the class explicitly:

In [47]:
class Task:
    completed = 0

    def mark_done(self):
        type(self).completed += 1

Let's test it:

In [48]:
t1 = Task()
t2 = Task()

t1.mark_done()
t2.mark_done()
t1.mark_done()

Task.completed

3

And the instances did not acquire local `completed` attributes:

In [49]:
t1.__dict__, t2.__dict__

({}, {})

### Solution

When state is intentionally shared, make the class-level update explicit.

That makes the code's intent much easier to reason about.

# Problem 7 — Default value plus optional per-instance override

This is a very common real-world design.

Suppose every API client uses a timeout of 30 seconds by default.

But some clients need a special timeout.

A good design is:

In [50]:
class APIClient:
    timeout = 30

    def __init__(self, name, timeout=None):
        self.name = name

        if timeout is not None:
            self.timeout = timeout

Notice that we do **not** copy `30` into every instance.

Let's create two clients:

In [51]:
normal = APIClient("normal")
fast = APIClient("fast", timeout=5)

Inspect the instance dictionaries:

In [52]:
normal.__dict__, fast.__dict__

({'name': 'normal'}, {'name': 'fast', 'timeout': 5})

The normal client stores only its name.

The fast client stores a local timeout because it is an actual override.

Now let's change the global default:

In [53]:
APIClient.timeout = 45

Predict:

In [54]:
normal.timeout, fast.timeout

(45, 5)

The normal client now sees `45`.

The fast client remains at `5`.

This is exactly what we wanted.

Now let's add a method for removing the override:

In [55]:
class APIClient:
    timeout = 30

    def __init__(self, name, timeout=None):
        self.name = name
        if timeout is not None:
            self.timeout = timeout

    def reset_timeout(self):
        if "timeout" in self.__dict__:
            del self.timeout

Let's test the full lifecycle:

In [56]:
client = APIClient("special", timeout=7)

print(client.timeout)
print(client.__dict__)

APIClient.timeout = 60
print(client.timeout)

client.reset_timeout()
print(client.timeout)
print(client.__dict__)

7
{'name': 'special', 'timeout': 7}
7
60
{'name': 'special'}


### Solution pattern

This is a powerful use of class attributes:

- class attribute = shared default,
- local instance attribute = explicit override,
- deleting the local attribute = return to default behavior.

# Problem 8 — Inspecting a class dictionary versus an instance dictionary

Let's create:

In [57]:
class Program:
    language = "Python"
    version = "3.x"

Now inspect the class dictionary:

In [58]:
Program.__dict__

mappingproxy({'__module__': '__main__',
              '__firstlineno__': 1,
              'language': 'Python',
              'version': '3.x',
              '__static_attributes__': (),
              '__dict__': <attribute '__dict__' of 'Program' objects>,
              '__weakref__': <attribute '__weakref__' of 'Program' objects>,
              '__doc__': None})

The class exposes a `mappingproxy`.

Let's confirm the type:

In [59]:
type(Program.__dict__)

mappingproxy

A mapping proxy gives us a read-only view of the class namespace.

So this will fail:

In [60]:
try:
    Program.__dict__["author"] = "Ada"
except TypeError as exc:
    print(type(exc).__name__ + ":", exc)

TypeError: 'mappingproxy' object does not support item assignment


But we *can* add a class attribute normally:

In [61]:
Program.author = "Ada"
Program.author

'Ada'

Now let's compare that with an instance:

In [62]:
p = Program()
type(p.__dict__), p.__dict__

(dict, {})

The instance dictionary is a real dictionary.

So technically we can modify it directly:

In [63]:
p.__dict__["edition"] = "community"
p.edition

'community'

That works.

But in normal application code, direct dictionary mutation is usually not the clearest interface.

Prefer:

In [64]:
p.release = "stable"
setattr(p, "build", 42)

p.__dict__

{'edition': 'community', 'release': 'stable', 'build': 42}

### Solution principle

Use `__dict__` mainly for:

- learning,
- introspection,
- debugging,
- framework/metaprogramming work.

Use normal attribute syntax for ordinary object updates.

# Problem 9 — Find out whether a value is local or inherited

Sometimes we can access an attribute, but we do not know where it came from.

Let's build a helper.

In [65]:
def explain_attribute(obj, name):
    instance_dict = getattr(obj, "__dict__", {})
    class_dict = type(obj).__dict__

    print(f"attribute: {name!r}")
    print("in instance __dict__?", name in instance_dict)
    print("in immediate class __dict__?", name in class_dict)

    if name in instance_dict:
        print("instance value:", instance_dict[name])

    if name in class_dict:
        print("class value:", class_dict[name])

    try:
        print("resolved value:", getattr(obj, name))
    except AttributeError:
        print("resolved value: <AttributeError>")

Now create a class and one instance:

In [66]:
class Account:
    apr = 2.5
    account_type = "Savings"

a = Account()
a.owner = "Ada"
a.apr = 1.0

Let's inspect three different kinds of attributes.

In [67]:
explain_attribute(a, "apr")

attribute: 'apr'
in instance __dict__? True
in immediate class __dict__? True
instance value: 1.0
class value: 2.5
resolved value: 1.0


Here `apr` exists in both places.

The instance value shadows the class value.

In [68]:
explain_attribute(a, "account_type")

attribute: 'account_type'
in instance __dict__? False
in immediate class __dict__? True
class value: Savings
resolved value: Savings


Here the name exists only on the class.

But the instance can still access it.

In [69]:
explain_attribute(a, "owner")

attribute: 'owner'
in instance __dict__? True
in immediate class __dict__? False
instance value: Ada
resolved value: Ada


Here the name exists only on the instance.

So this helper gives us a practical debugging technique:

> When a value is surprising, inspect both namespaces.

# Problem 10 — Inheritance adds another lookup layer

So far we have mostly looked at:

```text
instance -> class
```

But inheritance can add more levels.

Let's create a base class:

In [70]:
class BasePlan:
    discount = 5

And a subclass:

In [71]:
class GoldPlan(BasePlan):
    pass

Create an instance:

In [72]:
g = GoldPlan()

The instance has no local state:

In [73]:
g.__dict__

{}

The subclass also does not define `discount` directly:

In [74]:
"discount" in GoldPlan.__dict__

False

Yet this still works:

In [75]:
g.discount

5

Python finds `discount` on the base class.

Now let's change the base value:

In [76]:
BasePlan.discount = 10
g.discount

10

Now let's add an override to the subclass:

In [77]:
GoldPlan.discount = 20
g.discount

20

Now the subclass value hides the base-class value.

Let's add one more layer: an instance override.

In [78]:
g.discount = 30
g.discount

30

At this point the relevant lookup order is conceptually:

```text
instance -> subclass -> base class
```

Now let's peel the layers back one at a time.

In [79]:
del g.discount
g.discount

20

After deleting the instance value, the subclass value becomes visible again.

In [80]:
del GoldPlan.discount
g.discount

10

After deleting the subclass value, the base-class value becomes visible again.

### Solution

This is the same shadowing idea we already learned.

Inheritance simply gives Python more places to continue searching.

# Problem 11 — Refactoring repeated defaults

Consider this class:

In [81]:
class OldReport:
    def __init__(self, title):
        self.title = title
        self.format = "pdf"
        self.language = "en"
        self.page_size = "A4"

Every instance receives three identical values.

If these are truly defaults shared conceptually by all reports, we can move them to the class.

Let's refactor:

In [82]:
class Report:
    format = "pdf"
    language = "en"
    page_size = "A4"

    def __init__(self, title):
        self.title = title

Now create two reports:

In [83]:
r1 = Report("Quarterly")
r2 = Report("Annual")

Inspect their local state:

In [84]:
r1.__dict__, r2.__dict__

({'title': 'Quarterly'}, {'title': 'Annual'})

Only the unique title is stored locally.

Yet the defaults are still available:

In [85]:
r1.format, r1.language, r1.page_size

('pdf', 'en', 'A4')

Now give one report a language override:

In [86]:
r2.language = "fr"

And change the shared format:

In [87]:
Report.format = "html"

Let's inspect the results:

In [88]:
print(r1.format, r1.language)
print(r2.format, r2.language)
print(r1.__dict__)
print(r2.__dict__)

html en
html fr
{'title': 'Quarterly'}
{'title': 'Annual', 'language': 'fr'}


### Solution principle

Moving true defaults to the class can make the intended design clearer:

- shared default values stay on the class,
- unique state stays on the instance,
- exceptions are stored only where needed.

# Problem 12 — Safe bulk updates

Dynamic attributes are useful, but blindly assigning arbitrary names can be risky.

Suppose we receive:

In [89]:
incoming = {
    "theme": "dark",
    "language": "bg",
    "admin": True,
    "internal_token": "secret"
}

And our class is:

In [90]:
class Preferences:
    theme = "light"
    language = "en"

We want to allow updates to only `theme` and `language`.

Let's build a small update function.

In [91]:
def apply_allowed_updates(obj, updates, allowed):
    rejected = {}

    for name, value in updates.items():
        if name in allowed:
            setattr(obj, name, value)
        else:
            rejected[name] = value

    return rejected

Now apply the data:

In [92]:
prefs = Preferences()

rejected = apply_allowed_updates(
    prefs,
    incoming,
    allowed={"theme", "language"}
)

Inspect the instance:

In [93]:
prefs.__dict__

{'theme': 'dark', 'language': 'bg'}

And inspect the rejected values:

In [94]:
rejected

{'admin': True, 'internal_token': 'secret'}

Notice that the accepted values now shadow the class defaults:

In [95]:
Preferences.theme, prefs.theme, Preferences.language, prefs.language

('light', 'dark', 'en', 'bg')

### Solution principle

Dynamic assignment is powerful, but attribute names should often be validated before using `setattr`.

This is especially important when names come from external input.

# Problem 13 — Detect unnecessary shadows

Suppose:

In [96]:
class Feature:
    enabled = True

Now imagine an instance receives:

In [97]:
f = Feature()
f.enabled = True

That local value is identical to the class value.

Let's inspect:

In [98]:
f.__dict__

{'enabled': True}

This is not necessarily wrong.

But it is redundant.

More importantly, it prevents the instance from following future changes to the class default.

For example:

In [99]:
Feature.enabled = False

print("class:", Feature.enabled)
print("instance:", f.enabled)

class: False
instance: True


The instance remains `True` because it has a local shadow.

Let's write a function that removes a local shadow when it is identical to the current class value.

In [100]:
def remove_redundant_shadow(obj, name):
    if name not in obj.__dict__:
        return False

    cls = type(obj)

    if not hasattr(cls, name):
        return False

    if obj.__dict__[name] == getattr(cls, name):
        delattr(obj, name)
        return True

    return False

Let's test it with a fresh example:

In [101]:
class Mode:
    value = "safe"

m1 = Mode()
m2 = Mode()

m1.value = "safe"
m2.value = "fast"

print(remove_redundant_shadow(m1, "value"))
print(remove_redundant_shadow(m2, "value"))

print(m1.__dict__)
print(m2.__dict__)

True
False
{}
{'value': 'fast'}


`m1` now follows the class again.

`m2` keeps its meaningful override.

# Problem 14 — Capstone design: shared defaults, mutable instance state, and a shared counter

Now let's combine several ideas.

We want to build an `APIService` class with the following rules:

- every service has a `name`,
- every service has its own `headers` dictionary,
- the default timeout is shared,
- one service may override the timeout,
- all services share a request counter,
- removing a timeout override should restore the current class default.

Let's build it one decision at a time.

First, which values are naturally shared?

- `timeout`
- `request_count`

So those belong on the class.

In [102]:
class APIService:
    timeout = 30
    request_count = 0

Next, which values belong to individual objects?

- `name`
- `headers`

So they belong in `__init__`.

In [103]:
class APIService:
    timeout = 30
    request_count = 0

    def __init__(self, name):
        self.name = name
        self.headers = {}

Now add a method for setting one header.

Because `headers` belongs to the instance, modifying it affects only that service.

In [104]:
class APIService:
    timeout = 30
    request_count = 0

    def __init__(self, name):
        self.name = name
        self.headers = {}

    def set_header(self, key, value):
        self.headers[key] = value

Now add timeout override behavior.

Assigning `self.timeout` creates a local shadow.

In [105]:
class APIService:
    timeout = 30
    request_count = 0

    def __init__(self, name):
        self.name = name
        self.headers = {}

    def set_header(self, key, value):
        self.headers[key] = value

    def set_timeout(self, seconds):
        self.timeout = seconds

    def reset_timeout(self):
        if "timeout" in self.__dict__:
            del self.timeout

Finally, add the shared request counter.

We must be careful not to write:

```python
self.request_count += 1
```

because that would create an instance shadow.

Instead:

In [106]:
class APIService:
    timeout = 30
    request_count = 0

    def __init__(self, name):
        self.name = name
        self.headers = {}

    def set_header(self, key, value):
        self.headers[key] = value

    def set_timeout(self, seconds):
        self.timeout = seconds

    def reset_timeout(self):
        if "timeout" in self.__dict__:
            del self.timeout

    def record_request(self):
        type(self).request_count += 1

Now let's test the complete design.

In [107]:
alpha = APIService("alpha")
beta = APIService("beta")

alpha.set_header("Authorization", "token-A")
beta.set_header("Authorization", "token-B")

alpha.set_timeout(5)

alpha.record_request()
beta.record_request()
beta.record_request()

print("alpha headers:", alpha.headers)
print("beta headers:", beta.headers)

print("alpha timeout:", alpha.timeout)
print("beta timeout:", beta.timeout)

print("shared requests:", APIService.request_count)

print("alpha state:", alpha.__dict__)
print("beta state:", beta.__dict__)

alpha headers: {'Authorization': 'token-A'}
beta headers: {'Authorization': 'token-B'}
alpha timeout: 5
beta timeout: 30
shared requests: 3
alpha state: {'name': 'alpha', 'headers': {'Authorization': 'token-A'}, 'timeout': 5}
beta state: {'name': 'beta', 'headers': {'Authorization': 'token-B'}}


Now change the class default:

In [108]:
APIService.timeout = 60

print("alpha timeout:", alpha.timeout)
print("beta timeout:", beta.timeout)

alpha timeout: 5
beta timeout: 60


`alpha` keeps its local override.

`beta` follows the new shared default.

Now remove `alpha`'s override:

In [109]:
alpha.reset_timeout()

print("alpha timeout:", alpha.timeout)
print("alpha state:", alpha.__dict__)

alpha timeout: 60
alpha state: {'name': 'alpha', 'headers': {'Authorization': 'token-A'}}


Now both services follow the class default again.

Let's add a few assertions as a final correctness check.

In [110]:
assert alpha.headers == {"Authorization": "token-A"}
assert beta.headers == {"Authorization": "token-B"}
assert alpha.headers is not beta.headers

assert alpha.timeout == 60
assert beta.timeout == 60

assert APIService.request_count == 3
assert "request_count" not in alpha.__dict__
assert "request_count" not in beta.__dict__

print("Capstone checks passed.")

Capstone checks passed.


# Problem 15 — Final prediction challenge

We will finish with a compact problem.

Try to solve it on paper first.

Do not think only about values.

For every line, also ask:

> Is this value currently stored on the class or on the instance?

In [111]:
class Counter:
    value = 1

c1 = Counter()
c2 = Counter()

c1.value = Counter.value + 4
Counter.value = 10
c2.value = c1.value + Counter.value

print("Counter.value:", Counter.value)
print("c1.value:", c1.value)
print("c2.value:", c2.value)
print("c1 state:", c1.__dict__)
print("c2 state:", c2.__dict__)

Counter.value: 10
c1.value: 5
c2.value: 15
c1 state: {'value': 5}
c2 state: {'value': 15}


### Step-by-step solution

Initially:

```python
Counter.value == 1
```

Both instances have empty dictionaries.

Then:

```python
c1.value = Counter.value + 4
```

reads the class value `1`, computes `5`, and stores `5` locally on `c1`.

So now:

```text
Counter.value -> 1
c1.value      -> 5   # local
c2.value      -> 1   # class
```

Next:

```python
Counter.value = 10
```

changes the class value.

So:

```text
Counter.value -> 10
c1.value      -> 5   # still local
c2.value      -> 10  # still follows class
```

Finally:

```python
c2.value = c1.value + Counter.value
```

computes:

```python
5 + 10
```

and stores `15` locally on `c2`.

Therefore the final output is:

```text
Counter.value: 10
c1.value: 5
c2.value: 15
c1 state: {'value': 5}
c2 state: {'value': 15}
```

In [112]:
assert Counter.value == 10
assert c1.__dict__ == {"value": 5}
assert c2.__dict__ == {"value": 15}

print("Prediction verified.")

Prediction verified.


# Final Review

Let's summarize the most important ideas from all of the problems.

A class and its instances have separate state.

A class attribute does **not** automatically become a copied instance attribute when an object is created.

If an instance does not contain a requested name, Python can continue looking on the class.

If an instance later receives an attribute with the same name, that local value **shadows** the class value.

Deleting the local shadow can reveal the class value again.

This gives us a useful design pattern:

```text
class attribute     -> shared default
instance attribute  -> local override
delete override     -> return to shared default
```

There are also several practical pitfalls.

A mutable class attribute such as:

```python
items = []
```

is shared by every instance that resolves that name through the class.

That is correct only if sharing is intentional.

For per-instance mutable state, create the object in `__init__`.

Also be careful with expressions such as:

```python
self.count += 1
```

when `count` is intended to be shared.

The read may come from the class, but the write can create a new instance attribute.

Finally, remember the debugging workflow used throughout this notebook.

When an attribute surprises you:

1. inspect `obj.__dict__`,
2. inspect `type(obj).__dict__`,
3. check whether the name appears in one or both places,
4. identify whether shadowing is happening,
5. then reason about the lookup path.

With enough practice, class-vs-instance attribute behavior becomes predictable rather than mysterious.